# quorum on one GPU

quorum is a local re-implementation of the SystemOne request contract from TypeSafe's
[Jev](https://typesafe.ai): send one state and a set of typed questions (yes/no, choice,
score) and get each answer back with its probabilities, read from the model's token
logprobs. Here it runs on a 4B model on this instance's GPU. quorum is not affiliated
with TypeSafe.

The setup script started two services, both on loopback only (quorum has no auth):

- `quorum-llm`: llama-server with the model, on 127.0.0.1:8005
- `quorum`: the quorum shim in front of it, on 127.0.0.1:8017

How it compares, from the [full benchmark](https://github.com/sypherin/quorum/blob/master/docs/benchmark.md)
(5,327 scored units on 18 tasks): quorum on our judge model beats laya by 0.085 accuracy
and trails Jev by 0.221. Stock Qwen3-4B-Instruct-2507 behind the same shim trails Jev
by 0.117.

**Billing:** the instance bills by the hour while it runs. Stop it when you are done
(you then pay for storage only), or delete it.

In [ ]:
import json, os, shutil, statistics, subprocess, urllib.request
from pathlib import Path

def _app_dir():
    for p in (os.environ.get("QUORUM_APP"), "~/workspace/quorum", "/home/ubuntu/workspace/quorum"):
        if p and (Path(p).expanduser() / "quorum" / "serve.py").exists():
            return Path(p).expanduser()
    raise FileNotFoundError("quorum checkout not found: read ~/workspace/quorum-setup.log")

APP = _app_dir()
QUORUM = os.environ.get("QUORUM_URL", "http://127.0.0.1:8017")
LLM = os.environ.get("QUORUM_LLM_URL", "http://127.0.0.1:8005")

def get(url):
    with urllib.request.urlopen(url, timeout=10) as r:
        return json.loads(r.read() or b"null")

def ask(state, questions, **opts):
    body = json.dumps({"state": state, "questions": questions, **opts}).encode()
    req = urllib.request.Request(QUORUM + "/v1/systemone", body, {"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())

def show(r):
    print(f"model {r['model']} | {r['latency_ms']} ms | {r['calls']} upstream call(s) | cached {r['cached']}")
    for qid, a in r["answers"].items():
        if "error" in a:
            print(f"  {qid}: ERROR {a['error']}")
            continue
        committed = {"noul": a.get("answer"), "choice": a.get("choice"), "score": a.get("level")}[a["type"]]
        extra = ""
        if a["type"] == "noul" and a.get("noul") is not None:
            extra = f"  P(yes)={a['noul']:.3f}"
        elif a["type"] == "score":
            extra = f"  expected level={a['score']:.2f}"
        p = a.get("probabilities")
        dist = "  ".join(f"{k}={v:.3f}" for k, v in p.items()) if p else "no distribution (model fully confident)"
        print(f"  {qid} [{a['type']}] -> {committed}{extra}\n      {dist}")

print("quorum checkout:", APP)

## 1. Health

In [ ]:
if shutil.which("nvidia-smi"):
    print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total",
                                  "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
else:
    print("GPU: nvidia-smi not found")
print("llama-server:", get(LLM + "/health"))
print("model:", [m["id"] for m in get(LLM + "/v1/models")["data"]])
print("quorum:", get(QUORUM + "/healthz"))

## 2. One request, three primitives

The request has the same shape as a Jev SystemOne call: one `state`, then named questions.
`noul` is yes/no, `choice` picks one of the named options, `score` picks a level on an
ordered scale.

In [ ]:
r = ask(
    "Customer: I was charged twice for order 4471 and nobody has answered my emails in "
    "three days. I want the refund today.",
    {
        "urgent": {"type": "noul", "instructions": "Does this express urgency?"},
        "team": {"type": "choice", "instructions": "Which team should handle this?",
                 "criteria": {"billing": "payments, charges and refunds",
                              "technical": "bugs and errors in the product",
                              "sales": "plans and pricing"}},
        "mood": {"type": "score", "instructions": "How frustrated is the customer?",
                 "criteria": ["calm: factual", "annoyed: irritated", "angry: hostile"]},
    },
)
show(r)

Reading the answers:

- `P(yes)` on a yes/no question is always the probability of "yes", whichever side the
  model committed to (the Jev convention). The committed side is `answer`.
- `choice` probabilities cover the listed options only.
- On a score, `level` is the committed level and `score` is the expected level over the
  distribution.
- `probabilities` is null when the model is so sure that no other option reaches its top
  tokens.
- Raw probabilities run overconfident. `python3 -m quorum.calibrate labeled.jsonl` fits a
  small map from answers you have labeled (see the
  [README](https://github.com/sypherin/quorum#readme)).

## 3. Latency

Ten single-question requests, one at a time. Each uses a different state, so none of
them is served from quorum's answer cache.

In [ ]:
texts = [
    "The package arrived crushed and the screen is cracked.",
    "Thanks, the replacement works perfectly.",
    "Can I change the delivery address on my order?",
    "Third time asking: where is my refund?",
    "What are your opening hours on public holidays?",
    "The app logs me out every few minutes since the update.",
    "Great service, the technician was on time and polite.",
    "I was billed for a plan I cancelled last month.",
    "Do you ship to Malaysia?",
    "Your chatbot keeps sending me in circles and I am fed up.",
]
ms = []
for t in texts:
    r = ask(t, {"complaint": {"type": "noul", "instructions": "Is this a complaint?"}})
    assert not r["cached"], "served from cache: the latency would be meaningless"
    ms.append(r["latency_ms"])
print(f"median {statistics.median(ms):.0f} ms (min {min(ms)}, max {max(ms)}) over {len(ms)} requests")

For scale: on the integrated GPU the benchmark ran on, the median single-question item
took 482 ms with our judge and 304 ms with Qwen3-4B-Instruct-2507. Jev, called over the
network from Singapore, took 311 ms.

## 4. Rerun part of the benchmark

Three public tasks, one per primitive: SST-2 (yes/no, 150 items), AG News (choice, 300)
and SST-5 (score, 200). The cell downloads them from Hugging Face, asks quorum every item
and scores the answers. At full size it takes a few minutes on one GPU. Set `LIMIT` to a
small number for a quick look; the numbers then cover only the first `LIMIT` items of
each task and stop being comparable to the table below. This reruns quorum only: the
Jev and laya columns need their own setups.

Published accuracy, full run:

| task | Jev | laya | quorum + our judge | quorum + Qwen3-4B-2507 |
|---|---|---|---|---|
| sst2 | 0.960 | 0.620 | 0.793 | 0.907 |
| agnews | 0.843 | 0.923 | 0.853 | 0.837 |
| sst5 | 0.565 | 0.295 | 0.325 | 0.460 |

Expect your numbers near the column for the model you deployed, though not always equal
to it: the CUDA build here and the integrated GPU we measured on can settle a close call
differently.

In [ ]:
LIMIT = int(os.environ.get("BENCH_LIMIT", "0"))  # 0 = every item, comparable to the table
TASKS = ["sst2", "agnews", "sst5"]
PY = str(APP / ".venv" / "bin" / "python")
ARENA = APP / "bench" / "arena"
model = get(LLM + "/v1/models")["data"][0]["id"]
if "/" in model:  # it names the results directory
    raise RuntimeError(f"model id {model!r} is a file path: serve the model with --alias (setup.sh does)")
system = f"quorum-direct@{model}"  # results are kept per model, so a model switch never mixes them
OUT = APP.parent / f"quorum-report-{model}"
lim = ["--limit", str(LIMIT)] if LIMIT else []
env = {**os.environ, "ARENA_QUORUM_UPSTREAM": LLM}

def sh(*cmd):
    print("$", " ".join(Path(c).name if c.startswith("/") else c for c in cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=APP, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"exit {p.returncode}")

sh(PY, str(ARENA / "tasks.py"), *TASKS)
sh(PY, str(ARENA / "run.py"), system, *TASKS, *lim)
sh(PY, str(ARENA / "report.py"), system, "--tasks", ",".join(TASKS), *lim, "--out", str(OUT))
print("report saved:", OUT.with_suffix(".md"))

## 5. Switch the model, then stop

The launch parameter `QUORUM_MODEL` picks the model at deploy: `judge` (our
judgment-qc-gate-qwen3-4b, the default) or `qwen3-4b-2507`. To switch on a running
instance, open a terminal and run the setup script again:

```bash
curl -fsSL https://raw.githubusercontent.com/sypherin/quorum/master/deploy/brev/setup.sh | QUORUM_MODEL=qwen3-4b-2507 bash
```

Then rerun sections 1 to 4. When you are done, stop or delete the instance from the Brev
console.